In [1]:
import numpy as np
import pandas as pd

In [ ]:
main_df = pd.read_stata('/Users/veronicaupadhyay/Desktop/Python Projects/Machine Learning/main.dta')

conditions_df = pd.read_stata('/Users/veronicaupadhyay/Desktop/Python Projects/Machine Learning/conditions.dta') 
# one row [per person per condition]

In [ ]:
cond = conditions_df.copy()

code_cols = ['ICD10CDX', 'CCSR1X', 'CCSR2X', 'CCSR3X', 'CCSR4X'] # 

for c in code_cols:
    cond[c] = cond[c].astype(str).str.strip().str.upper()

# ICD-10 F10–F19 (substance-related)
print(cond.loc[cond.ICD10CDX.str.match(r'^F1\d$'), 'ICD10CDX'].value_counts())

# CCSR substance-related categories (check these labels in the codebook)
ccsr_vals = pd.Series(cond[code_cols[1:]].values.ravel())
print(ccsr_vals[ccsr_vals.str.match(r'^MBD0(1[7-9]|2[0-6])$')].value_counts())

ICD10CDX
F19    52
Name: count, dtype: int64
MBD025    52
Name: count, dtype: int64


There's 52 conditions reported... not a lot. How many people is it really?

In [8]:
icd_map = {'F10': 'alcohol', 'F11': 'opioid', 'F12': 'cannabis', 'F13': 'sedative',
           'F14': 'stimulant', 'F15': 'stimulant', 'F16': 'other', 'F17': 'tobacco',
           'F18': 'other', 'F19': 'other'}
ccsr_map = {'MBD017': 'alcohol', 'MBD018': 'opioid', 'MBD019': 'cannabis',
            'MBD020': 'sedative', 'MBD021': 'stimulant', 'MBD022': 'other',
            'MBD023': 'other', 'MBD024': 'tobacco', 'MBD025': 'other'}

cond['sud_type'] = cond.ICD10CDX.map(icd_map)
for c in code_cols[1:]:
    cond['sud_type'] = cond['sud_type'].fillna(cond[c].map(ccsr_map))

sud = cond.dropna(subset=['sud_type'])
person = pd.crosstab(sud.DUPERSID, sud.sud_type).clip(upper=1).add_prefix('sud_')
person['sud_nontobacco'] = person.drop(columns='sud_tobacco', errors='ignore').max(axis=1)
person['sud_any'] = 1
person = person.reset_index()
#---

main_df['DUPERSID'] = main_df['DUPERSID'].astype(str).str.strip()
person['DUPERSID'] = person['DUPERSID'].astype(str).str.strip()

df = main_df.merge(person, on='DUPERSID', how='left')
sud_cols = [c for c in df.columns if c.startswith('sud_')]
df[sud_cols] = df[sud_cols].fillna(0).astype(int)

age = pd.to_numeric(df["AGEY1X"].astype("string"), errors="coerce")
expense = pd.to_numeric(df["TOTEXPY2"].astype("string"), errors="coerce")

adults = df.loc[(age >= 18) & (expense >= 0)].copy()  # also drops negative missing codes
print(len(adults))
print(adults[sud_cols].sum().sort_values(ascending=False))

# --- 
adults['smoker_y1'] = (adults.ADSMOK2 == 1).astype(int)   # 1 = yes, 2 = no; negatives are missing
adults['sud_broad'] = ((adults.sud_any == 1) | (adults.smoker_y1 == 1)).astype(int)
print(adults[['sud_any', 'smoker_y1', 'sud_broad']].sum())

# ---
sud_rows = cond[cond.ICD10CDX.eq('F19') | cond[code_cols[1:]].eq('MBD025').any(axis=1)]

print("rows:", len(sud_rows))
print("people:", sud_rows.DUPERSID.nunique())
print(sud_rows.PANEL.value_counts())

in_panel = adults[adults.DUPERSID.isin(sud_rows.DUPERSID)]
print("adults in your longitudinal panel:", len(in_panel))

7079
sud_other         23
sud_nontobacco    23
sud_any           23
dtype: int64
sud_any      23
smoker_y1     0
sud_broad    23
dtype: int64
rows: 52
people: 51
PANEL
28 PANEL 28    28
27 PANEL 27    24
Name: count, dtype: int64
adults in your longitudinal panel: 23


In [10]:
def code(s):
    """Extract the leading numeric code from MEPS labeled values like '1 YES' or '-1 INAPPLICABLE'."""
    return pd.to_numeric(
        s.astype('string').str.extract(r'^\s*(-?\d+(?:\.\d+)?)')[0],
        errors='coerce'
    )

adults['smoker_y1'] = (code(adults['ADSMOK2']) == 1).astype(int)
adults['sud_broad'] = ((adults.sud_any == 1) | (adults.smoker_y1 == 1)).astype(int)
print(adults[['sud_any', 'smoker_y1', 'sud_broad']].sum())

sud_any       23
smoker_y1    570
sud_broad    585
dtype: int64
